# Class 1 — What Is a Large Language Model?
**Week 2: Introduction to LLMs — "The Black Box / Neural Signal"**

### Learning objectives
By the end of this notebook you will be able to:
- Trace the pre-LLM history of getting machines to "understand" language: rule-based, statistical, neural, transformer
- Build a tiny, from-scratch demo of each era's core idea and see why each one gave way to the next
- Build a toy next-token predictor using word-frequency counts
- Connect the toy demos to what a real LLM does at inference time

Run each cell in order with **Shift+Enter**. No API key needed — every demo in this notebook runs on toy data, locally.

## Setup
No external packages beyond `numpy` are required for this class.

In [1]:
import sys
import numpy as np

print(f"Python version: {sys.version.split()[0]}")
print("Setup complete — no API key needed for this notebook.")

Python version: 3.12.13
Setup complete — no API key needed for this notebook.


## 1. A Short History of Getting Machines to Understand Language
Four eras, one thread: each new approach fixed a real limitation of the one before it. We'll build a tiny, honest demo of each — not a real system, just enough code to feel the underlying idea.

### 1.1 Rule-based era: pattern matching, not understanding
The earliest chatbots didn't predict anything — they matched keywords against hand-written rules and picked a canned response. ELIZA (1966) is the classic example: it played a Rogerian therapist by echoing your words back as a question.

In [2]:
import re

RULES = [
    (r"\bI (feel|am) (.*)", "Why do you {0} {1}?"),
    (r"\bmy (mother|father|mom|dad)\b", "Tell me more about your family."),
    (r"\bI need (.*)", "Why do you need {0}?"),
]

def eliza_reply(text):
    for pattern, template in RULES:
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            return template.format(*m.groups())
    return "Tell me more about that."

for line in ["I feel tired", "my mother called today", "I need a break", "the weather is nice"]:
    print(f"you:   {line}")
    print(f"eliza: {eliza_reply(line)}\n")

you:   I feel tired
eliza: Why do you feel tired?

you:   my mother called today
eliza: Tell me more about your family.

you:   I need a break
eliza: Why do you need a break?

you:   the weather is nice
eliza: Tell me more about that.



`eliza_reply` doesn't know what "tired" means — it just rearranges your own words into a question. Say something with none of these keywords ("the weather is nice") and you fall through to the generic default. That fallback line *is* the ceiling of rule-based systems: every input the author didn't anticipate needs its own hand-written rule.

### 1.2 Statistical era: learn patterns from a corpus
By the 1990s the field moved from hand-written rules to counting: given a labeled body of text, which words show up in which category? A "bag of words" classifier below scores a message as spam-like or not, purely from word overlap — no rules, no grammar, just counts.

In [3]:
from collections import Counter

spam_words = set("free money now click win prize offer".split())
ham_words = set("meeting schedule report project update team".split())

def bag_of_words_score(text, positive_words, negative_words):
    words = text.lower().split()
    pos = sum(1 for w in words if w in positive_words)
    neg = sum(1 for w in words if w in negative_words)
    return pos - neg

messages = [
    "click now to win free money",
    "let's schedule the project meeting",
    "free prize offer, click to win",
]
for msg in messages:
    score = bag_of_words_score(msg, spam_words, ham_words)
    label = "SPAM" if score > 0 else "not spam"
    print(f"{label:9} (score {score:+d}) — {msg!r}")

SPAM      (score +5) — 'click now to win free money'
not spam  (score -3) — "let's schedule the project meeting"
SPAM      (score +4) — 'free prize offer, click to win'


This "bag of words" throws away word order entirely — "dog bites man" and "man bites dog" score identically. That's the statistical era's core trade-off: more general than hand-written rules, but it treats language as an unordered pile of evidence, not a structure.

### 1.3 Neural era: words become vectors
Word embeddings (word2vec, GloVe, ~2013) represent each word as a point in space, learned so that words used in similar contexts land near each other. The famous party trick: vector arithmetic captures relationships — "king" minus "man" plus "woman" lands near "queen". Below is a hand-built miniature of that idea. Real embeddings have hundreds of dimensions, learned from billions of words; these four are made up by hand, just to show the shape of the idea.

In [4]:
# hand-set toy vectors — real embeddings are learned, not hand-written
vectors = {
    "king":  np.array([0.9, 0.8, 0.1]),
    "man":   np.array([0.8, 0.2, 0.1]),
    "woman": np.array([0.2, 0.2, 0.1]),
    "queen": np.array([0.3, 0.8, 0.1]),
}

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

result = vectors["king"] - vectors["man"] + vectors["woman"]
print("king - man + woman ≈", result.round(2))
print()
for word, vec in vectors.items():
    print(f"similarity to result: {word:6} -> {cosine_sim(result, vec):.3f}")

king - man + woman ≈ [0.3 0.8 0.1]

similarity to result: king   -> 0.885
similarity to result: man    -> 0.574
similarity to result: woman  -> 0.891
similarity to result: queen  -> 1.000


"queen" comes out nearest — the toy vectors capture a real relationship (gender, as a direction in space) without ever being told the rule "a queen is a female king." This geometric view of meaning is what later gets fed into transformers.

### 1.4 Transformer era: attention over vectors
Word embeddings alone are static — "bank" gets the same vector whether you mean a riverbank or a financial bank. Self-attention lets each word's representation shift based on its neighbors. Below is a tiny numpy version: dot products between word vectors, turned into weights with softmax — the same core computation a transformer's attention layer runs, just with random (not learned) numbers.

In [5]:
words_a = ["deposited", "cash", "at", "the", "bank"]
words_b = ["sat", "by", "the", "river", "bank"]

np.random.seed(0)
base = {w: np.random.randn(4) for w in set(words_a + words_b) - {"bank"}}
base["bank"] = np.random.randn(4)

def attention_weights(words, vectors):
    vecs = np.array([vectors[w] for w in words])
    scores = vecs @ vecs.T
    exp = np.exp(scores - scores.max(axis=1, keepdims=True))
    return exp / exp.sum(axis=1, keepdims=True)

for words in (words_a, words_b):
    weights = attention_weights(words, base)
    bank_idx = words.index("bank")
    print(f"sentence: {' '.join(words)}")
    for w, weight in zip(words, weights[bank_idx]):
        print(f"  'bank' attends to {w:10} -> {weight:.2f}")
    print()

sentence: deposited cash at the bank
  'bank' attends to deposited  -> 0.04
  'bank' attends to cash       -> 0.00
  'bank' attends to at         -> 0.02
  'bank' attends to the        -> 0.05
  'bank' attends to bank       -> 0.89

sentence: sat by the river bank
  'bank' attends to sat        -> 0.02
  'bank' attends to by         -> 0.38
  'bank' attends to the        -> 0.03
  'bank' attends to river      -> 0.02
  'bank' attends to bank       -> 0.55



The vectors here are random, not learned — so don't read meaning into which word "wins" in this toy. What matters is the mechanism: attention is dot products between vectors, turned into weights by softmax. A real transformer learns vectors where "deposited" and "cash" genuinely pull "bank" toward its financial meaning — this is the same computation, just with meaningful numbers instead of random ones.

## 2. Next-Token Prediction, Toy Version
Same idea as Section 1.2's spam counter, aimed at a different question: instead of counting which words label a message as spam, count which word tends to *follow* another. That's already most of what a next-token predictor does.

In [6]:
from collections import defaultdict, Counter

corpus = "the cat sat the cat ran the dog sat"
follows = defaultdict(Counter)
words = corpus.split()
for a, b in zip(words, words[1:]):
    follows[a][b] += 1

for word, counter in follows.items():
    print(f"{word:6} -> {dict(counter)}")

def predict_next(word):
    counter = follows.get(word)
    if not counter:
        return None
    return counter.most_common(1)[0][0]

print("\nPredict next after 'the':", predict_next("the"))

the    -> {'cat': 2, 'dog': 1}
cat    -> {'sat': 1, 'ran': 1}
sat    -> {'the': 1}
ran    -> {'the': 1}
dog    -> {'sat': 1}

Predict next after 'the': cat


### Week 2, Class 1 — closed
Four eras, one thread: each new approach fixed a limitation of the last. Rules couldn't scale past what someone anticipated. Counting patterns ignored order and meaning. Static vectors couldn't adapt to context. Attention did. Stack enough attention layers, train on enough text, and you get the model behind the rest of this week's slide deck.

## Challenges
Work through these in order. No solutions are provided — each starter cell has a `# TODO` marking where your code goes.

### Challenge 1 — Extend the Corpus
Add at least 3 more sentences to the `corpus` string from Section 2 (reuse the same words plus at least one new word), rebuild `follows`, and print the new `follows['the']` counter plus `predict_next('the')`.

**Acceptance criteria:** your corpus has at least 4 sentences total; you print the updated `follows['the']` counter and the new prediction.

In [10]:
# TODO: extend corpus, rebuild follows, print follows['the'] and predict_next('the')
from collections import defaultdict, Counter

# A larger training corpus
corpus = """
the cat sat on the mat
the dog sat near the cat
the student opened the book
the teacher entered the classroom
the programmer wrote the code
the doctor treated the patient
the chef cooked the food
the child played in the garden
the bird flew over the tree
the car stopped near the house
"""

# Convert the corpus into lowercase words
words = corpus.lower().split()

# Store which words follow each word
follows = defaultdict(Counter)

# Learn word-to-next-word relationships
for current_word, next_word in zip(words, words[1:]):
    follows[current_word][next_word] += 1


def predict_next(word):
    """Predict the most common word that follows the given word."""
    word = word.lower()

    choices = follows.get(word)

    if not choices:
        return None

    return choices.most_common(1)[0][0]


# Display everything that followed "the"
print('Words following "the":')
print(follows["the"])

# Test predictions
print('\nPrediction after "the":', predict_next("the"))
print('Prediction after "student":', predict_next("student"))
print('Prediction after "programmer":', predict_next("programmer"))
print('Prediction after "doctor":', predict_next("doctor"))

Words following "the":
Counter({'cat': 2, 'mat': 1, 'dog': 1, 'student': 1, 'book': 1, 'teacher': 1, 'classroom': 1, 'programmer': 1, 'code': 1, 'doctor': 1, 'patient': 1, 'chef': 1, 'food': 1, 'child': 1, 'garden': 1, 'bird': 1, 'tree': 1, 'car': 1, 'house': 1})

Prediction after "the": cat
Prediction after "student": opened
Prediction after "programmer": wrote
Prediction after "doctor": treated


### Challenge 2 — Chain Predictions Into a Phrase
Write a function `generate(start_word, steps)` that calls `predict_next` repeatedly, appending each predicted word, and stops early if a word has no known follower. Run it starting from `'the'` for 5 steps.

**Acceptance criteria:** returns a list of words starting with `start_word`, length at most `steps + 1`, and stops cleanly (no crash) when `predict_next` returns `None`.

In [11]:
# TODO: write generate(start_word, steps) using predict_next, then call generate('the', 5)
def generate(start_word, steps):
    """
    Generate a phrase by repeatedly predicting the next word.

    Parameters:
        start_word: The first word of the phrase.
        steps: Number of next-word predictions to make.
    """
    current_word = start_word.lower()
    generated_words = [current_word]

    for _ in range(steps):
        next_word = predict_next(current_word)

        # Stop when the model does not know what comes next
        if next_word is None:
            break

        generated_words.append(next_word)
        current_word = next_word

    return " ".join(generated_words)


# Test the generator
print(generate("the", 5))
print(generate("student", 4))
print(generate("programmer", 4))

the cat sat on the cat
student opened the cat sat
programmer wrote the cat sat


### Challenge 3 — Break the Toy Predictor
Call `predict_next` on a word that never appears in your corpus (e.g. `'giraffe'`). Print what happens, then modify `predict_next` so it returns a readable string like `"<no data>"` instead of `None` for unseen words.

**Acceptance criteria:** calling `predict_next` on an out-of-vocabulary word no longer returns a bare `None` — it returns a readable message.

In [13]:
# TODO: call predict_next on an unseen word, then update it to return a readable message instead of None
print("\nBefore improving predict_next:")
print(predict_next("giraffe"))


def predict_next(word):
    """Return a prediction or a readable error message."""
    word = word.lower().strip()

    choices = follows.get(word)

    if not choices:
        return f"No prediction available: '{word}' was not found in the corpus."

    return choices.most_common(1)[0][0]


def generate_safe(start_word, steps):
    """Generate text and safely handle unknown words."""
    current_word = start_word.lower().strip()

    if current_word not in follows:
        return f"Cannot generate text: '{current_word}' was not found in the corpus."

    generated_words = [current_word]

    for _ in range(steps):
        next_word = predict_next(current_word)

        if next_word.startswith("No prediction available:"):
            break

        generated_words.append(next_word)
        current_word = next_word

    return " ".join(generated_words)


print("\nCHALLENGE 3")
print("-" * 40)
print(predict_next("giraffe"))
print(generate_safe("the", 5))
print(generate_safe("giraffe", 5))


Before improving predict_next:
None

CHALLENGE 3
----------------------------------------
No prediction available: 'giraffe' was not found in the corpus.
the cat sat on the cat
Cannot generate text: 'giraffe' was not found in the corpus.


### Challenge 4 — Connect the Toy to Real LLMs
Write 2-4 sentences (right in this cell, replacing the italic prompt) answering: (a) what `follows` and a real LLM's parameters have in common, and (b) one thing a real LLM can do that this toy model structurally cannot, and why. (Hint: think about corpus size vs. training-set size, and single-word context vs. long context.)

The toy predictor and a real Large Language Model both predict what is likely
to come next based on patterns learned from text. However, the toy predictor
only counts which complete word followed another word in a very small corpus.
A real LLM learns from a massive amount of data, works mainly with tokens, and
uses neural networks to consider the wider context of the sentence. Therefore,
a real LLM can generate much more meaningful, flexible, and coherent responses.